In [36]:
!pip install shap lime xgboost

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 6.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=e7e301292cd05f7007e242e2ae073686bde5ad639b88f22c7fd41e6896b89f75
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
Successfully built lime


In [37]:
import pandas as pd
import numpy as np
import shap

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

In [38]:
from google.colab import files
uploaded = files.upload()

Saving train.csv to train (3).csv


In [39]:
df = pd.read_csv('train.csv')

# Drop missing values
df = df.dropna()

# Encode categorical columns
le = LabelEncoder()
for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col])

df.head()

,Gender,Married,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Loan_Status
0,1,1,1,1,22852,7939,187,240,1,1
1,0,0,1,0,23801,5565,221,120,0,0
2,1,1,0,0,15084,4544,435,360,1,1
3,1,1,0,0,13644,4976,62,120,1,1
4,1,0,1,1,15897,3419,217,360,1,1


In [40]:
X = df.drop('Loan_Status', axis=1)
y = df['Loan_Status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [41]:
model = XGBClassifier()
model.fit(X_train, y_train)

preds = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, preds))

Accuracy: 0.975


In [42]:
explainer = shap.Explainer(model)
shap_values = explainer(X_test)

In [43]:
def explain_prediction(index):
    data_point = X_test.iloc[index]
    shap_val = shap_values[index].values

    feature_impact = dict(zip(X.columns, shap_val))

    # Sort features by impact
    sorted_features = sorted(feature_impact.items(), key=lambda x: abs(x[1]), reverse=True)

    top_features = sorted_features[:3]

    return top_features

In [44]:
def generate_reasoning(top_features):
    reasoning = []

    for feature, impact in top_features:
        if impact > 0:
            reasoning.append(f"{feature} positively influenced approval")
        else:
            reasoning.append(f"{feature} negatively influenced approval")

    return reasoning

In [45]:
def consistency_check(shap_features, reasoning):
    consistent = True

    for (feature, impact), reason in zip(shap_features, reasoning):
        if impact > 0 and "negatively" in reason:
            consistent = False
        if impact < 0 and "positively" in reason:
            consistent = False

    return consistent

In [51]:
def self_explaining_ai(index):
    try:
        prediction = model.predict([X_test.iloc[index]])[0]
        confidence = np.max(model.predict_proba([X_test.iloc[index]]))

        shap_features = explain_prediction(index)
        reasoning = generate_reasoning(shap_features)
        consistency = consistency_check(shap_features, reasoning)

        result = "Approved ✅" if prediction == 1 else "Rejected ❌"

        report = f"""
🔍 DECISION REPORT
================================

Prediction: {result}
Confidence: {round(confidence*100,2)}%

Top Reasons:
"""

        for f, val in shap_features:
            report += f"\n- {f} (impact: {round(val,3)})"

        report += "\n\nLLM-style Reasoning:\n"
        for r in reasoning:
            report += f"- {r}\n"

        report += "\nConsistency Check:\n"
        report += "✔ Consistent" if consistency else "❌ Inconsistent"

        return report

    except Exception as e:
        return f"Error: {str(e)}"

In [52]:
import gradio as gr

def run_model(index):
    index = int(index)
    return self_explaining_ai(index)

interface = gr.Interface(
    fn=run_model,
    inputs=gr.Number(label="Enter Test Data Index (e.g., 0–100)"),
    outputs=gr.Textbox(label="AI Decision Report"),
    title="🧠 Self-Explaining AI (XAI + Reasoning)",
    description="Enter an index to see prediction + explanation"
)

In [58]:
import matplotlib.pyplot as plt
import shap
import matplotlib.pyplot as plt

def plot_shap(index):
    try:
        row = X_test.iloc[index:index+1]

        shap_values = explainer.shap_values(row)

        # Handle both outputs
        if isinstance(shap_values, list):
            shap_val = shap_values[1][0]
            base_val = explainer.expected_value[1]
        else:
            shap_val = shap_values[0]
            base_val = explainer.expected_value

        plt.figure()

        shap.waterfall_plot(
            shap.Explanation(
                values=shap_val,
                base_values=base_val,
                data=row.iloc[0],
                feature_names=X_test.columns.tolist()
            )
        )

        plt.savefig("shap_plot.png", bbox_inches='tight')
        plt.close()

        return "shap_plot.png"

    except Exception as e:
        return None   # IMPORTANT: prevent crash

In [65]:
def run_model(index):
    index = int(index)

    report = self_explaining_ai(index)
    plot = plot_shap(index)

    if plot is None:
        return report, None
    else:
        return report, plot

In [66]:
interface = gr.Interface(
    fn=run_model,
    inputs=gr.Slider(0, len(X_test)-1, step=1, label="Select Data Index"),
    outputs=[
        gr.Textbox(label="AI Decision Report"),
        gr.Image(label="SHAP Explanation")
    ],
    title="🧠 Self-Explaining AI System",
    description="Prediction + Reasoning + SHAP Visualization"
)

In [67]:
interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8cfcf50f096d26fc35.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
